## 1. Promotions Data Prep

This transforms the Key Dates (Promotions) schedule into a pre-processed set of dummy variables which is joined to both the training and the prediction datasets when models are run.

Main predictors are promo_name and promo_discount.

This is only applied to Weekly models, so we group on weekly data granulaity only.

In [1]:
# notebook config
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

# packages
import pandas as pd
import numpy as np

# functions
import functions.transform as tf

In [2]:
# data load & transform
df_promos = tf.clean_df(pd.read_csv(f"../data/meta/promo_calendar_202609.csv"))
df_promos = df_promos[['promotions','brand','top_discount','start','end']]
df_promos['start'] = pd.to_datetime(df_promos['start'])
df_promos['end'] = pd.to_datetime(df_promos['end'])

#qa
df_promos.head()
df_promos.info()

<class 'pandas.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   promotions    161 non-null    str           
 1   brand         161 non-null    str           
 2   top_discount  157 non-null    float64       
 3   start         161 non-null    datetime64[us]
 4   end           161 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 12.0 KB


/var/folders/sq/nh5_91qn4n7dggks6m77sgc40000gp/T/ipykernel_48848/378993373.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_promos['start'] = pd.to_datetime(df_promos['start'])
/var/folders/sq/nh5_91qn4n7dggks6m77sgc40000gp/T/ipykernel_48848/378993373.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_promos['end'] = pd.to_datetime(df_promos['end'])


In [3]:
# Define date range and brands
start_date = pd.to_datetime('2024-02-05') # +1 day
end_date = pd.to_datetime('2027-01-31')
brands = ['brand us', 'brand outlet', 'brand ca']

# Adjust start_date to the previous Sunday if not already a Sunday
start_sunday = start_date - pd.Timedelta(days=start_date.weekday() + 1)

# Create the weekly date range
weekly_dates = pd.date_range(start=start_sunday, end=end_date, freq='W-SUN')

# Create df_weeks with a row for each week-brand combination
df_weeks_list = []
for brand in brands:
    df_temp = pd.DataFrame({'weekstart': weekly_dates, 'brand': brand})
    df_weeks_list.append(df_temp)

df_weeks = pd.concat(df_weeks_list, ignore_index=True)

# qa
df_weeks.head()

,weekstart,brand
0,2024-02-04,brand us
1,2024-02-11,brand us
2,2024-02-18,brand us
3,2024-02-25,brand us
4,2024-03-03,brand us


In [4]:
# # BINARY PROMO FLAGS (commented out - using overlap_days instead)
# # create promo flags
# 
# # For each promotion-brand combination, check if the week overlaps
# for idx, promo in df_promos.iterrows():
#     promo_name = promo['promotions']
#     promo_brand = promo['brand']
#     
#     # Skip rows with missing data
#     if pd.isna(promo_name) or pd.isna(promo_brand):
#         continue
#     
#     col_name = f"promo_{promo_name.lower().replace(' ', '_')}"
#     
#     # Initialize column with 0 if it doesn't exist yet
#     if col_name not in df_weeks.columns:
#         df_weeks[col_name] = 0
#     
#     # Create a mask for matching brand AND overlapping date range
#     mask = (
#         (df_weeks['brand'] == promo_brand) &
#         (df_weeks['weekstart'] <= promo['end']) & 
#         (df_weeks['weekstart'] + pd.Timedelta(days=6) >= promo['start'])
#     )
#     
#     # Set to 1 only where mask is True
#     df_weeks.loc[mask, col_name] = 1

In [5]:
# # BINARY DISCOUNT FLAGS (commented out - using overlap_days instead)
# # create discount flags
# 
# # For each discount-brand combination, check if the week overlaps
# for idx, promo in df_promos.iterrows():
#     promo_brand = promo['brand']
#     discount = promo['top_discount']
#     
#     # Skip rows with missing data
#     if pd.isna(discount) or pd.isna(promo_brand):
#         continue
#     
#     # Clean the discount value for column name
#     discount_str = str(discount).replace('%', '').replace('.', '_').strip()
#     col_name = f"discount_{discount_str}"
#     
#     # Initialize column with 0 if it doesn't exist yet
#     if col_name not in df_weeks.columns:
#         df_weeks[col_name] = 0
#     
#     # Create a mask for matching brand AND overlapping date range
#     mask = (
#         (df_weeks['brand'] == promo_brand) &
#         (df_weeks['weekstart'] <= promo['end']) & 
#         (df_weeks['weekstart'] + pd.Timedelta(days=6) >= promo['start'])
#     )
#     
#     # Set to 1 only where mask is True
#     df_weeks.loc[mask, col_name] = 1

In [6]:
# # CREATE PROMO & DISCOUNT FLAGS USING OVERLAP DAYS (0-7)
# # (commented out for now, more unstable than dummies)
# # Integer count of days each promo/discount overlaps with each week
# # Replaces binary dummies to better handle partial-week promotions

# def calc_overlap_days(week_start, promo_start, promo_end):
#     """Calculate number of days a promo overlaps with a given week (0-7)"""
#     week_end = week_start + pd.Timedelta(days=6)
#     overlap_start = max(week_start, promo_start)
#     overlap_end = min(week_end, promo_end)
#     if overlap_start <= overlap_end:
#         return (overlap_end - overlap_start).days + 1
#     return 0

# # Create promo overlap columns
# for idx, promo in df_promos.iterrows():
#     promo_name = promo['promotions']
#     promo_brand = promo['brand']
#     promo_start = promo['start']
#     promo_end = promo['end']
    
#     if pd.isna(promo_name) or pd.isna(promo_brand):
#         continue
    
#     col_name = f"promo_{promo_name.lower().replace(' ', '_')}"
    
#     if col_name not in df_weeks.columns:
#         df_weeks[col_name] = 0
    
#     mask = df_weeks['brand'] == promo_brand
    
#     for week_idx in df_weeks[mask].index:
#         week_start = df_weeks.loc[week_idx, 'weekstart']
#         overlap_days = calc_overlap_days(week_start, promo_start, promo_end)
#         df_weeks.loc[week_idx, col_name] = max(df_weeks.loc[week_idx, col_name], overlap_days)

# # Create discount overlap columns
# for idx, promo in df_promos.iterrows():
#     promo_brand = promo['brand']
#     discount = promo['top_discount']
#     promo_start = promo['start']
#     promo_end = promo['end']
    
#     if pd.isna(discount) or pd.isna(promo_brand):
#         continue
    
#     discount_str = str(discount).replace('%', '').replace('.', '_').strip()
#     col_name = f"discount_{discount_str}"
    
#     if col_name not in df_weeks.columns:
#         df_weeks[col_name] = 0
    
#     mask = df_weeks['brand'] == promo_brand
    
#     for week_idx in df_weeks[mask].index:
#         week_start = df_weeks.loc[week_idx, 'weekstart']
#         overlap_days = calc_overlap_days(week_start, promo_start, promo_end)
#         df_weeks.loc[week_idx, col_name] = max(df_weeks.loc[week_idx, col_name], overlap_days)

# # QA: Check results
# print("Promo/Discount columns now use overlap days (0-7):")
# promo_cols = [c for c in df_weeks.columns if c.startswith('promo_')][:3]
# discount_cols = [c for c in df_weeks.columns if c.startswith('discount_')][:3]
# df_weeks[['weekstart', 'brand'] + promo_cols + discount_cols].head(20)

In [7]:
# CREATE PROMO & DISCOUNT FLAGS USING BUCKETED EXPOSURE
# Ordinal encoding: 0=none, 1=partial (1-3 days), 2=full (4-7 days)
# Avoids linearity assumption while capturing partial-week effect

def calc_overlap_days(week_start, promo_start, promo_end):
    """Calculate number of days a promo overlaps with a given week (0-7)"""
    week_end = week_start + pd.Timedelta(days=6)
    overlap_start = max(week_start, promo_start)
    overlap_end = min(week_end, promo_end)
    if overlap_start <= overlap_end:
        return (overlap_end - overlap_start).days + 1
    return 0

def days_to_bucket(overlap_days):
    """Convert overlap days to bucket: 0=none, 1=partial, 2=full"""
    if overlap_days == 0:
        return 0
    elif overlap_days <= 3:
        return 1
    else:
        return 2

# Create promo bucketed columns
for idx, promo in df_promos.iterrows():
    promo_name = promo['promotions']
    promo_brand = promo['brand']
    promo_start = promo['start']
    promo_end = promo['end']
    
    if pd.isna(promo_name) or pd.isna(promo_brand):
        continue
    
    col_name = f"promo_{promo_name.lower().replace(' ', '_')}"
    
    if col_name not in df_weeks.columns:
        df_weeks[col_name] = 0
    
    mask = df_weeks['brand'] == promo_brand
    
    for week_idx in df_weeks[mask].index:
        week_start = df_weeks.loc[week_idx, 'weekstart']
        overlap_days = calc_overlap_days(week_start, promo_start, promo_end)
        bucket = days_to_bucket(overlap_days)
        df_weeks.loc[week_idx, col_name] = max(df_weeks.loc[week_idx, col_name], bucket)

# Create discount bucketed columns
for idx, promo in df_promos.iterrows():
    promo_brand = promo['brand']
    discount = promo['top_discount']
    promo_start = promo['start']
    promo_end = promo['end']
    
    if pd.isna(discount) or pd.isna(promo_brand):
        continue
    
    discount_str = str(discount).replace('%', '').replace('.', '_').strip()
    col_name = f"discount_{discount_str}"
    
    if col_name not in df_weeks.columns:
        df_weeks[col_name] = 0
    
    mask = df_weeks['brand'] == promo_brand
    
    for week_idx in df_weeks[mask].index:
        week_start = df_weeks.loc[week_idx, 'weekstart']
        overlap_days = calc_overlap_days(week_start, promo_start, promo_end)
        bucket = days_to_bucket(overlap_days)
        df_weeks.loc[week_idx, col_name] = max(df_weeks.loc[week_idx, col_name], bucket)

# QA: Check bucket distribution
print("Bucketed Exposure values (0=none, 1=partial, 2=full):")
promo_cols = [c for c in df_weeks.columns if c.startswith('promo_')][:3]
discount_cols = [c for c in df_weeks.columns if c.startswith('discount_')][:3]
df_weeks[['weekstart', 'brand'] + promo_cols + discount_cols].head(20)

Bucketed Exposure values (0=none, 1=partial, 2=full):


,weekstart,brand,promo_presidents_day,promo_friends_&_family,promo_easter_pre-peak,discount_0_5,discount_0_4,discount_0_7
0,2024-02-04,brand us,0,0,0,0,0,0
1,2024-02-11,brand us,0,0,0,0,0,0
2,2024-02-18,brand us,0,0,0,0,0,0
3,2024-02-25,brand us,0,0,0,0,0,0
4,2024-03-03,brand us,0,1,0,0,1,0
5,2024-03-10,brand us,0,2,0,0,2,0
6,2024-03-17,brand us,0,0,0,0,0,0
7,2024-03-24,brand us,0,0,0,0,0,0
8,2024-03-31,brand us,0,0,0,0,0,0
9,2024-04-07,brand us,0,0,0,0,0,0


In [8]:
# save results - be sure to manually QA
df_weeks.to_csv(f"../data/train/promo_dummies.csv", index=False)